# Real Estate Scraper - Three Sites

This notebook scrapes real estate listings from:

- **Wassit**: `http://wassit.info/immobilier.html`
- **L'agence MR**: `lagence-mr.com/`
- **OpenSooq MR**: `mr.opensooq.com/`

It outputs a cleaned **pandas DataFrame** and exports it to `raw.csv`.

> ⚠️ Scraping note: Respect each website's Terms of Service and robots.txt, keep a reasonable rate-limit, and don't run this in a tight loop.

In [12]:
# Core imports
import re
import time
from typing import Optional, Dict, Any, List

import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

In [13]:
# Networking helpers: requests Session with retries + a browser-like User-Agent
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from urllib3.exceptions import InsecureRequestWarning
import warnings

# Suppress SSL warnings for sites with certificate issues
warnings.filterwarnings('ignore', category=InsecureRequestWarning)

DEFAULT_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9,fr;q=0.8,ar;q=0.7",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

def make_session(
    total_retries: int = 5,
    backoff_factor: float = 0.6,
    status_forcelist: tuple = (429, 500, 502, 503, 504),
    verify_ssl: bool = False,  # Set to False for sites with SSL issues
) -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=total_retries,
        connect=total_retries,
        read=total_retries,
        status=total_retries,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        allowed_methods=frozenset(["GET", "HEAD"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.verify = verify_ssl
    return session

def fetch_html(session: requests.Session, url: str, timeout: int = 20) -> str:
    r = session.get(url, headers=DEFAULT_HEADERS, timeout=timeout, verify=False)
    r.raise_for_status()
    return r.text

## Wassit Scraper

Scrapes listings from `http://wassit.info/immobilier.html`

In [14]:
# Wassit parsing
WASSIT_URL = "http://wassit.info/immobilier.html"
WASSIT_BASE_URL = "http://wassit.info"

_price_re_wassit = re.compile(r"(\d[\d\s]*)\s*UM", re.IGNORECASE)

def parse_wassit_html(html: str, base_url: str = WASSIT_BASE_URL) -> pd.DataFrame:
    """Parse Wassit real estate listings page."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    
    # Look for announcement blocks - adjust selectors based on actual HTML structure
    # Common patterns: div with class containing "annonce", "listing", "ad", etc.
    listings = []
    
    # Try multiple selector patterns
    selectors = [
        "div.center",
        "div.annonce",
        "div.listing",
        "table tr",
        "div[class*='annonce']",
        "div[class*='listing']",
    ]
    
    for selector in selectors:
        elements = soup.select(selector)
        if elements:
            listings = elements
            print(f"Found {len(listings)} listings using selector: {selector}")
            break
    
    # If no specific selector works, try finding all links that might be listings
    if not listings:
        # Look for links in announcement sections
        for a in soup.find_all("a", href=True):
            href = a.get("href", "")
            if "/annonces/" in href or "/immobilier" in href.lower():
                parent = a.find_parent(["div", "tr", "td"])
                if parent:
                    listings.append(parent)
    
    for item in listings:
        try:
            # Get text content
            text = item.get_text(" ", strip=True)
            if not text or len(text) < 10:  # Skip very short texts
                continue
            
            # Find URL
            url = None
            a_tag = item.find("a", href=True)
            if a_tag:
                href = a_tag.get("href", "")
                if href.startswith("http"):
                    url = href
                else:
                    url = urljoin(base_url, href)
            
            # Extract price
            price = None
            price_match = _price_re_wassit.search(text)
            if price_match:
                try:
                    price_str = price_match.group(1).replace(" ", "").replace(",", "")
                    price = int(price_str)
                except:
                    pass
            
            # Extract title (usually first line or link text)
            title = None
            if a_tag:
                title = a_tag.get_text(strip=True)
            if not title:
                # Try to get first meaningful line
                lines = [line.strip() for line in text.split("\n") if line.strip()]
                title = lines[0] if lines else text[:100]
            
            # Extract location (common locations in Mauritania)
            location = None
            locations = ["Nouakchott", "Nouadhibou", "Nouakchott - Mauritanie", "Nouadhibou - Mauritanie"]
            for loc in locations:
                if loc in text:
                    location = loc.replace(" - Mauritanie", "")
                    break
            
            rows.append({
                "source": "wassit",
                "url": url,
                "title": title,
                "raw_text": text[:500],  # Limit raw text length
                "price_mru": price,
                "location": location,
            })
        except Exception as e:
            continue
    
    df = pd.DataFrame(rows)
    if not df.empty and "url" in df.columns:
        df = df.drop_duplicates(subset=["url"], keep="first")
    return df

# Scrape Wassit
print("Scraping Wassit...")
session = make_session(verify_ssl=False)
wassit_df = pd.DataFrame()
try:
    time.sleep(1.0)
    wassit_html = fetch_html(session, WASSIT_URL, timeout=25)
    wassit_df = parse_wassit_html(wassit_html)
    print(f"Wassit listings found: {len(wassit_df)}")
    if not wassit_df.empty:
        display(wassit_df.head())
except Exception as e:
    print(f"Wassit fetch/parse failed: {repr(e)}")
    print("Continuing with other sites...")

Scraping Wassit...
Found 20 listings using selector: div.center
Wassit listings found: 20


,source,url,title,raw_text,price_mru,location
0,wassit,http://wassit.info/annonces/12434.html,Maison aux Lvelouje,14 500 000 UM Maison aux Lvelouje Ù†ÙˆØ§ÙƒØ´...,14500000.0,None
1,wassit,http://wassit.info/annonces/12406.html,"Machine de brique, parpaing, pavÃ©s autobloquant","Machine de brique, parpaing, pavÃ©s autobloqua...",NaN,Nouakchott
2,wassit,http://wassit.info/annonces/12405.html,Machine de fabrication de brique,Machine de fabrication de brique Nouakchott - ...,NaN,Nouakchott
3,wassit,http://wassit.info/annonces/12403.html,Machine a parpaing,Machine a parpaing Nouakchott - Mauritanie Vu ...,NaN,Nouakchott
4,wassit,http://wassit.info/annonces/12375.html,COMPLEXE A VENDRE,725 000 UM COMPLEXE A VENDRE Nouadhibou - Maur...,725000.0,Nouadhibou


## L'agence MR Scraper

Scrapes listings from `lagence-mr.com/`

In [15]:
# L'agence MR parsing
LAGENCE_URL = "https://lagence-mr.com/"
LAGENCE_BASE_URL = "https://lagence-mr.com"

_price_re_lagence = re.compile(r"(\d[\d\s,]*)\s*(?:MRU|UM|MRO)", re.IGNORECASE)

def parse_lagence_html(html: str, base_url: str = LAGENCE_BASE_URL) -> pd.DataFrame:
    """Parse L'agence MR real estate listings page."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    
    # Try to find listing containers
    listings = []
    selectors = [
        "div.property",
        "div.listing",
        "div.annonce",
        "article",
        "div[class*='property']",
        "div[class*='listing']",
        "div[class*='card']",
        "div[class*='item']",
    ]
    
    for selector in selectors:
        elements = soup.select(selector)
        if elements and len(elements) > 2:  # At least a few listings
            listings = elements
            print(f"Found {len(listings)} listings using selector: {selector}")
            break
    
    # If still no listings, try finding property links
    if not listings:
        for a in soup.find_all("a", href=True):
            href = a.get("href", "").lower()
            if any(keyword in href for keyword in ["property", "immobilier", "bien", "annonce", "listing"]):
                parent = a.find_parent(["div", "article", "li"])
                if parent:
                    listings.append(parent)
    
    for item in listings:
        try:
            text = item.get_text(" ", strip=True)
            if not text or len(text) < 10:
                continue
            
            # Find URL
            url = None
            a_tag = item.find("a", href=True)
            if a_tag:
                href = a_tag.get("href", "")
                if href.startswith("http"):
                    url = href
                else:
                    url = urljoin(base_url, href)
            
            # Extract price
            price = None
            price_match = _price_re_lagence.search(text)
            if price_match:
                try:
                    price_str = price_match.group(1).replace(" ", "").replace(",", "")
                    price = int(price_str)
                except:
                    pass
            
            # Extract title
            title = None
            if a_tag:
                title = a_tag.get_text(strip=True)
            if not title:
                h_tag = item.find(["h1", "h2", "h3", "h4"])
                if h_tag:
                    title = h_tag.get_text(strip=True)
            if not title:
                lines = [line.strip() for line in text.split("\n") if line.strip()]
                title = lines[0] if lines else text[:100]
            
            # Extract location
            location = None
            locations = ["Nouakchott", "Nouadhibou", "Tevragh Zeina", "Dar Naim", "Arafat"]
            for loc in locations:
                if loc in text:
                    location = loc
                    break
            
            rows.append({
                "source": "lagence-mr",
                "url": url,
                "title": title,
                "raw_text": text[:500],
                "price_mru": price,
                "location": location,
            })
        except Exception as e:
            continue
    
    df = pd.DataFrame(rows)
    if not df.empty and "url" in df.columns:
        df = df.drop_duplicates(subset=["url"], keep="first")
    return df

# Scrape L'agence MR
print("\nScraping L'agence MR...")
lagence_df = pd.DataFrame()
try:
    time.sleep(1.0)
    lagence_html = fetch_html(session, LAGENCE_URL, timeout=25)
    lagence_df = parse_lagence_html(lagence_html)
    print(f"L'agence MR listings found: {len(lagence_df)}")
    if not lagence_df.empty:
        display(lagence_df.head())
except Exception as e:
    print(f"L'agence MR fetch/parse failed: {repr(e)}")
    print("Continuing with other sites...")


Scraping L'agence MR...
Found 27 listings using selector: div[class*='listing']
L'agence MR listings found: 9


,source,url,title,raw_text,price_mru,location
0,lagence-mr,https://lagence-mr.com/bien/appartement-meuble...,Appartement meublé haut standing 70000 MRU A l...,Appartement meublé haut standing 70000 MRU A l...,70000,None
5,lagence-mr,https://lagence-mr.com/bien/duplexe-meuble-a-l...,Duplexe meublé à louer 80000 MRU A louer 2 Cha...,Duplexe meublé à louer 80000 MRU A louer 2 Cha...,80000,None
7,lagence-mr,https://lagence-mr.com/bien/maison-spacieuse-a...,Maison spacieuse à louer 50000 MRU A louer Tev...,Maison spacieuse à louer 50000 MRU A louer Tev...,50000,None
9,lagence-mr,https://lagence-mr.com/bien/maison-a-louer-3/,Maison à louer 30000 MRU A louer ModuleE 2 Cha...,Maison à louer 30000 MRU A louer ModuleE 2 Cha...,30000,None
11,lagence-mr,https://lagence-mr.com/bien/maison-a-louer-2/,Maison à louer 60000 MRU A louer Tevragh zeina...,Maison à louer 60000 MRU A louer Tevragh zeina...,60000,None


In [16]:
# OpenSooq MR parsing
OPENSOOQ_URL = "https://mr.opensooq.com/"
OPENSOOQ_BASE_URL = "https://mr.opensooq.com"

# Try to find the real estate section
OPENSOOQ_REAL_ESTATE_URLS = [
    "https://mr.opensooq.com/mauritania/all?s[main_category]=real_estate",
    "https://mr.opensooq.com/mauritania/all?s[main_category]=real-estate",
    "https://mr.opensooq.com/mauritania/real-estate",
    "https://mr.opensooq.com/mauritania/all?category_id=real_estate",
]

_price_re_opensooq = re.compile(r"(\d[\d\s,]*)\s*(?:MRU|UM|MRO|دولار|ريال)", re.IGNORECASE)

def parse_opensooq_html(html: str, base_url: str = OPENSOOQ_BASE_URL) -> pd.DataFrame:
    """Parse OpenSooq MR real estate listings page."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    
    # OpenSooq typically uses specific class names
    listings = []
    selectors = [
        "div.postItem",
        "div.post-item",
        "div.listing-item",
        "div.item",
        "div[class*='post']",
        "div[class*='listing']",
        "div[class*='ad']",
        "article",
    ]
    
    for selector in selectors:
        elements = soup.select(selector)
        if elements and len(elements) > 2:
            listings = elements
            print(f"Found {len(listings)} listings using selector: {selector}")
            break
    
    # Alternative: find all links that might be listings
    if not listings:
        for a in soup.find_all("a", href=True):
            href = a.get("href", "").lower()
            if "/post/" in href or "/listing/" in href or "/ad/" in href:
                parent = a.find_parent(["div", "article", "li"])
                if parent:
                    listings.append(parent)
    
    for item in listings:
        try:
            text = item.get_text(" ", strip=True)
            if not text or len(text) < 10:
                continue
            
            # Find URL
            url = None
            a_tag = item.find("a", href=True)
            if a_tag:
                href = a_tag.get("href", "")
                if href.startswith("http"):
                    url = href
                else:
                    url = urljoin(base_url, href)
            
            # Extract price
            price = None
            price_match = _price_re_opensooq.search(text)
            if price_match:
                try:
                    price_str = price_match.group(1).replace(" ", "").replace(",", "")
                    price = int(price_str)
                except:
                    pass
            
            # Extract title
            title = None
            if a_tag:
                title = a_tag.get_text(strip=True)
            if not title:
                h_tag = item.find(["h1", "h2", "h3", "h4", "h5"])
                if h_tag:
                    title = h_tag.get_text(strip=True)
            if not title:
                # Try to find title in class names
                title_elem = item.find(class_=re.compile(r"title|name|heading", re.I))
                if title_elem:
                    title = title_elem.get_text(strip=True)
            if not title:
                lines = [line.strip() for line in text.split("\n") if line.strip()]
                title = lines[0] if lines else text[:100]
            
            # Extract location
            location = None
            locations = ["Nouakchott", "Nouadhibou", "Tevragh Zeina", "Dar Naim", "Arafat", "نواكشوط"]
            for loc in locations:
                if loc in text:
                    location = loc
                    break
            
            rows.append({
                "source": "opensooq-mr",
                "url": url,
                "title": title,
                "raw_text": text[:500],
                "price_mru": price,
                "location": location,
            })
        except Exception as e:
            continue
    
    df = pd.DataFrame(rows)
    if not df.empty and "url" in df.columns:
        df = df.drop_duplicates(subset=["url"], keep="first")
    return df

# Scrape OpenSooq MR
print("\nScraping OpenSooq MR...")
opensooq_df = pd.DataFrame()

# Try multiple URLs
for url in [OPENSOOQ_URL] + OPENSOOQ_REAL_ESTATE_URLS:
    try:
        time.sleep(1.5)
        opensooq_html = fetch_html(session, url, timeout=25)
        opensooq_df = parse_opensooq_html(opensooq_html)
        if not opensooq_df.empty:
            print(f"OpenSooq MR listings found: {len(opensooq_df)} (from {url})")
            display(opensooq_df.head())
            break
    except Exception as e:
        print(f"Tried {url}: {repr(e)}")
        continue

if opensooq_df.empty:
    print("OpenSooq MR: No listings found. The site structure may need adjustment.")


Scraping OpenSooq MR...
Tried https://mr.opensooq.com/mauritania/all?s[main_category]=real_estate: HTTPError('410 Client Error: Gone for url: https://mr.opensooq.com/mauritania/ar/mauritania/all?s%5Bmain_category%5D=real_estate')
Tried https://mr.opensooq.com/mauritania/all?s[main_category]=real-estate: HTTPError('410 Client Error: Gone for url: https://mr.opensooq.com/mauritania/ar/mauritania/all?s%5Bmain_category%5D=real-estate')
Tried https://mr.opensooq.com/mauritania/real-estate: HTTPError('410 Client Error: Gone for url: https://mr.opensooq.com/mauritania/ar/mauritania/real-estate')
Tried https://mr.opensooq.com/mauritania/all?category_id=real_estate: HTTPError('410 Client Error: Gone for url: https://mr.opensooq.com/mauritania/ar/mauritania/all?category_id=real_estate')
OpenSooq MR: No listings found. The site structure may need adjustment.


## Detail Page Scrapers

Functions to extract detailed information from individual listing pages.

In [17]:
# Helper functions for extracting detailed fields

PROPERTY_TYPES = ["Villa", "Appartement", "Terrain", "Duplex", "Studio", "Maison", "Riad"]
LISTING_TYPES = ["Vente", "Location", "vente", "location", "louer", "vendre", "à louer", "à vendre"]

# Regex patterns
_price_pattern = re.compile(r"(\d[\d\s,]*)\s*(?:MRU|UM|MRO)", re.IGNORECASE)
_surface_pattern = re.compile(r"(\d+)\s*(?:m²|m2|m\s*²|metres?|mètres?)", re.IGNORECASE)
_chambres_pattern = re.compile(r"(\d+)\s*(?:chambre|chambres|bedroom|bedrooms)", re.IGNORECASE)
_salons_pattern = re.compile(r"(\d+)\s*(?:salon|salons|living|living room)", re.IGNORECASE)
_sdb_pattern = re.compile(r"(\d+)\s*(?:salle?\s*de?\s*bain|sdb|bathroom|bathrooms|wc)", re.IGNORECASE)
_date_pattern = re.compile(r"(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})", re.IGNORECASE)

# Features keywords
FEATURES_KEYWORDS = {
    "piscine": ["piscine", "pool", "swimming"],
    "garage": ["garage", "parking"],
    "climatisation": ["climatisation", "air conditionné", "a/c", "ac", "climatiseur"],
    "jardin": ["jardin", "garden", "yard"],
    "terrasse": ["terrasse", "terrace", "balcon", "balcony"],
    "ascenseur": ["ascenseur", "elevator", "lift"],
    "sécurité": ["sécurité", "security", "garde", "vigilance"],
    "meublé": ["meublé", "furnished", "meublée"],
}

def extract_property_type(text: str) -> Optional[str]:
    """Extract property type from text."""
    text_lower = text.lower()
    for prop_type in PROPERTY_TYPES:
        if prop_type.lower() in text_lower:
            # Normalize
            if prop_type.lower() == "maison":
                return "Villa"  # Maison often means Villa
            if prop_type.lower() == "riad":
                return "Villa"  # Riad is a type of Villa
            return prop_type
    return None

def extract_listing_type(text: str) -> Optional[str]:
    """Extract listing type (Vente or Location)."""
    text_lower = text.lower()
    if any(keyword in text_lower for keyword in ["location", "louer", "à louer", "rent"]):
        return "Location"
    if any(keyword in text_lower for keyword in ["vente", "vendre", "à vendre", "sale", "sell"]):
        return "Vente"
    return None

def extract_features(text: str) -> List[str]:
    """Extract features/characteristics from text."""
    text_lower = text.lower()
    features = []
    for feature, keywords in FEATURES_KEYWORDS.items():
        if any(keyword in text_lower for keyword in keywords):
            features.append(feature)
    return features

def extract_number(pattern: re.Pattern, text: str) -> Optional[int]:
    """Extract a number using a regex pattern."""
    match = pattern.search(text)
    if match:
        try:
            return int(match.group(1).replace(" ", "").replace(",", ""))
        except:
            pass
    return None

def parse_wassit_detail(html: str) -> Dict[str, Any]:
    """Parse Wassit detail page."""
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(" ", strip=True)
    
    result = {
        "titre": None,
        "type_bien": None,
        "type_annonce": None,
        "prix": None,
        "surface_m2": None,
        "nb_chambres": None,
        "nb_salons": None,
        "nb_sdb": None,
        "quartier": None,
        "ville": None,
        "description": None,
        "date_publication": None,
        "caracteristiques": [],
    }
    
    # Extract title - look for announcement title, not page title
    # Try to find the actual listing title in the content area
    title_candidates = []
    # Look for h2 or h3 in main content
    for h in soup.find_all(["h2", "h3"]):
        h_text = h.get_text(strip=True)
        if h_text and len(h_text) > 5 and "WASSIT" not in h_text.upper():
            title_candidates.append(h_text)
    
    # Look for strong/bold text that might be the title
    for strong in soup.find_all(["strong", "b"]):
        strong_text = strong.get_text(strip=True)
        if strong_text and len(strong_text) > 10 and len(strong_text) < 200:
            if "WASSIT" not in strong_text.upper() and "annonces" not in strong_text.lower():
                title_candidates.append(strong_text)
    
    # Look in table cells or divs with specific patterns
    for td in soup.find_all("td"):
        td_text = td.get_text(strip=True)
        if td_text and len(td_text) > 10 and len(td_text) < 200:
            if any(keyword in td_text.lower() for keyword in ["maison", "appartement", "villa", "terrain", "duplex"]):
                title_candidates.append(td_text)
                break
    
    if title_candidates:
        result["titre"] = title_candidates[0]
    else:
        # Fallback: use h1 but filter out page titles
        title_elem = soup.find("h1")
        if title_elem:
            title_text = title_elem.get_text(strip=True)
            if "WASSIT" not in title_text.upper() and len(title_text) < 200:
                result["titre"] = title_text
    
    # Extract property type
    result["type_bien"] = extract_property_type(text)
    
    # Extract listing type
    result["type_annonce"] = extract_listing_type(text)
    
    # Extract price
    price_match = _price_pattern.search(text)
    if price_match:
        try:
            result["prix"] = int(price_match.group(1).replace(" ", "").replace(",", ""))
        except:
            pass
    
    # Extract surface
    result["surface_m2"] = extract_number(_surface_pattern, text)
    
    # Extract bedrooms
    result["nb_chambres"] = extract_number(_chambres_pattern, text)
    
    # Extract living rooms
    result["nb_salons"] = extract_number(_salons_pattern, text)
    
    # Extract bathrooms
    result["nb_sdb"] = extract_number(_sdb_pattern, text)
    
    # Extract location (quartier and ville)
    locations = ["Tevragh Zeina", "Dar Naim", "Arafat", "Teyarett", "Ksar", "Riyadh", 
                "Las Palmas", "FNord", "ENord", "ModuleA", "ModuleE", "Sekter"]
    for loc in locations:
        if loc in text:
            result["quartier"] = loc
            break
    
    if "Nouakchott" in text:
        result["ville"] = "Nouakchott"
    elif "Nouadhibou" in text:
        result["ville"] = "Nouadhibou"
    
    # Extract description (main content) - exclude navigation and headers
    desc_elem = soup.find("div", class_=re.compile(r"description|content|detail", re.I))
    if desc_elem:
        desc_text = desc_elem.get_text(" ", strip=True)
        # Remove common page elements
        desc_text = re.sub(r"WASSIT\.info.*?Mauritanie", "", desc_text, flags=re.IGNORECASE)
        desc_text = re.sub(r"Qui sommes nous.*?Règlement", "", desc_text, flags=re.DOTALL)
        result["description"] = desc_text[:2000].strip()
    else:
        # Try to find main content area
        main_content = soup.find("div", class_=re.compile(r"center|main|content", re.I))
        if main_content:
            desc_text = main_content.get_text(" ", strip=True)
            # Clean up
            desc_text = re.sub(r"WASSIT\.info.*?Mauritanie", "", desc_text, flags=re.IGNORECASE)
            result["description"] = desc_text[:2000].strip()
        else:
            # Use a reasonable portion of text, excluding headers
            clean_text = re.sub(r"WASSIT\.info.*?Mauritanie", "", text, flags=re.IGNORECASE)
            result["description"] = clean_text[:2000].strip() if len(clean_text) > 100 else clean_text.strip()
    
    # Extract date
    date_match = _date_pattern.search(text)
    if date_match:
        result["date_publication"] = date_match.group(1)
    
    # Extract features
    result["caracteristiques"] = extract_features(text)
    
    return result

def parse_lagence_detail(html: str) -> Dict[str, Any]:
    """Parse L'agence MR detail page."""
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(" ", strip=True)
    
    result = {
        "titre": None,
        "type_bien": None,
        "type_annonce": None,
        "prix": None,
        "surface_m2": None,
        "nb_chambres": None,
        "nb_salons": None,
        "nb_sdb": None,
        "quartier": None,
        "ville": None,
        "description": None,
        "date_publication": None,
        "caracteristiques": [],
    }
    
    # Extract title - prioritize h1 in main content
    title_elem = soup.find("h1")
    if title_elem:
        title_text = title_elem.get_text(strip=True)
        # Filter out generic page titles
        if "L'agence" not in title_text and len(title_text) > 5:
            result["titre"] = title_text
    if not result["titre"]:
        # Try h2
        for h2 in soup.find_all("h2"):
            h2_text = h2.get_text(strip=True)
            if len(h2_text) > 5 and len(h2_text) < 200:
                result["titre"] = h2_text
                break
    
    # Extract property type
    result["type_bien"] = extract_property_type(text)
    
    # Extract listing type
    result["type_annonce"] = extract_listing_type(text)
    
    # Extract price
    price_match = _price_pattern.search(text)
    if price_match:
        try:
            result["prix"] = int(price_match.group(1).replace(" ", "").replace(",", ""))
        except:
            pass
    
    # Extract surface
    result["surface_m2"] = extract_number(_surface_pattern, text)
    
    # Extract bedrooms (look for "X Chambres" pattern)
    result["nb_chambres"] = extract_number(_chambres_pattern, text)
    
    # Extract living rooms
    result["nb_salons"] = extract_number(_salons_pattern, text)
    
    # Extract bathrooms (look for "X Salle de bain" pattern)
    result["nb_sdb"] = extract_number(_sdb_pattern, text)
    
    # Extract location
    locations = ["Tevragh Zeina", "Dar Naim", "Arafat", "Las Palmas", "FNord", "ENord", 
                "ModuleA", "ModuleE", "Sekter", "Teyarett"]
    for loc in locations:
        if loc in text:
            result["quartier"] = loc
            break
    
    result["ville"] = "Nouakchott"  # L'agence MR is primarily Nouakchott
    
    # Extract description
    desc_elem = soup.find("div", class_=re.compile(r"description|content|detail", re.I))
    if desc_elem:
        result["description"] = desc_elem.get_text(" ", strip=True)[:2000]
    else:
        result["description"] = text[:2000] if len(text) > 100 else text
    
    # Extract date
    date_match = _date_pattern.search(text)
    if date_match:
        result["date_publication"] = date_match.group(1)
    
    # Extract features
    result["caracteristiques"] = extract_features(text)
    
    return result

def parse_opensooq_detail(html: str) -> Dict[str, Any]:
    """Parse OpenSooq MR detail page."""
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(" ", strip=True)
    
    result = {
        "titre": None,
        "type_bien": None,
        "type_annonce": None,
        "prix": None,
        "surface_m2": None,
        "nb_chambres": None,
        "nb_salons": None,
        "nb_sdb": None,
        "quartier": None,
        "ville": None,
        "description": None,
        "date_publication": None,
        "caracteristiques": [],
    }
    
    # Extract title
    title_elem = soup.find("h1") or soup.find("h2") or soup.find(class_=re.compile(r"title|titre", re.I))
    if title_elem:
        result["titre"] = title_elem.get_text(strip=True)
    
    # Extract property type
    result["type_bien"] = extract_property_type(text)
    
    # Extract listing type
    result["type_annonce"] = extract_listing_type(text)
    
    # Extract price
    price_match = _price_pattern.search(text)
    if price_match:
        try:
            result["prix"] = int(price_match.group(1).replace(" ", "").replace(",", ""))
        except:
            pass
    
    # Extract surface
    result["surface_m2"] = extract_number(_surface_pattern, text)
    
    # Extract bedrooms
    result["nb_chambres"] = extract_number(_chambres_pattern, text)
    
    # Extract living rooms
    result["nb_salons"] = extract_number(_salons_pattern, text)
    
    # Extract bathrooms
    result["nb_sdb"] = extract_number(_sdb_pattern, text)
    
    # Extract location
    locations = ["Tevragh Zeina", "Dar Naim", "Arafat", "Teyarett", "Ksar", "Riyadh"]
    for loc in locations:
        if loc in text:
            result["quartier"] = loc
            break
    
    if "Nouakchott" in text or "نواكشوط" in text:
        result["ville"] = "Nouakchott"
    elif "Nouadhibou" in text or "نواذيبو" in text:
        result["ville"] = "Nouadhibou"
    
    # Extract description
    desc_elem = soup.find("div", class_=re.compile(r"description|content|detail|post", re.I))
    if desc_elem:
        result["description"] = desc_elem.get_text(" ", strip=True)[:2000]
    else:
        result["description"] = text[:2000] if len(text) > 100 else text
    
    # Extract date
    date_match = _date_pattern.search(text)
    if date_match:
        result["date_publication"] = date_match.group(1)
    
    # Extract features
    result["caracteristiques"] = extract_features(text)
    
    return result

## Enrich Listings with Detail Pages

Fetch detail pages for each listing to extract all required fields.

In [18]:
def enrich_listings_with_details(
    df: pd.DataFrame,
    session: requests.Session,
    delay_s: float = 1.0,
    max_rows: Optional[int] = None,
) -> pd.DataFrame:
    """Enrich listings DataFrame with details from individual pages."""
    if df.empty or "url" not in df.columns:
        return df
    
    urls = df["url"].dropna().drop_duplicates().tolist()
    if max_rows is not None:
        urls = urls[:max_rows]
    
    print(f"Enriching {len(urls)} listings with detail pages...")
    
    url_to_details = {}
    for i, url in enumerate(urls):
        try:
            html = fetch_html(session, url, timeout=15)
            
            # Determine which parser to use based on source
            source = df[df["url"] == url]["source"].iloc[0] if "source" in df.columns else None
            
            if source == "wassit":
                details = parse_wassit_detail(html)
            elif source == "lagence-mr":
                details = parse_lagence_detail(html)
            elif source == "opensooq-mr":
                details = parse_opensooq_detail(html)
            else:
                # Try to detect from URL
                if "wassit" in url.lower():
                    details = parse_wassit_detail(html)
                elif "lagence" in url.lower():
                    details = parse_lagence_detail(html)
                elif "opensooq" in url.lower():
                    details = parse_opensooq_detail(html)
                else:
                    details = {}
            
            url_to_details[url] = details
            
        except Exception as e:
            url_to_details[url] = {}
            if (i + 1) % 10 == 0:
                print(f"  Processed {i + 1}/{len(urls)} (errors: {sum(1 for d in url_to_details.values() if not d)})")
        
        time.sleep(delay_s)
        
        if (i + 1) % 50 == 0:
            print(f"  Enriched {i + 1}/{len(urls)} listings...")
    
    # Add detail columns to DataFrame
    detail_columns = [
        "titre", "type_bien", "type_annonce", "prix", "surface_m2",
        "nb_chambres", "nb_salons", "nb_sdb", "quartier", "ville",
        "description", "date_publication", "caracteristiques"
    ]
    
    for col in detail_columns:
        if col == "caracteristiques":
            # Store as comma-separated string for CSV
            df[col] = df["url"].map(
                lambda u: ", ".join(url_to_details.get(u, {}).get(col, [])) if url_to_details.get(u, {}).get(col) else None
            )
        else:
            df[col] = df["url"].map(lambda u: url_to_details.get(u, {}).get(col))
    
    # Fill in missing values from existing columns (only if detail extraction failed)
    if "titre" in df.columns and "title" in df.columns:
        # Only fill if titre is None or generic page title
        mask = df["titre"].isna() | df["titre"].astype(str).str.contains("WASSIT|L'agence|Annonces", case=False, na=False)
        df.loc[mask, "titre"] = df.loc[mask, "title"]
    
    if "prix" in df.columns and "price_mru" in df.columns:
        df["prix"] = df["prix"].fillna(df["price_mru"])
    
    if "ville" in df.columns and "location" in df.columns:
        df["ville"] = df["ville"].fillna(df["location"])
    
    # Clean up titles - remove generic page titles
    if "titre" in df.columns:
        df["titre"] = df["titre"].astype(str).str.replace(r"WASSIT\.info.*?Mauritanie", "", case=False, regex=True)
        df["titre"] = df["titre"].astype(str).str.replace(r"L'agence.*", "", case=False, regex=True)
        df["titre"] = df["titre"].str.strip()
        df["titre"] = df["titre"].replace(["", "nan", "None"], None)
    
    print(f"✅ Enriched {len([u for u, d in url_to_details.items() if d])} listings with details.")
    return df

# Set to None to enrich all listings, or a number for testing (e.g., 10)
ENRICH_MAX_ROWS = None  # Change to 10 for quick testing

## Combine and Export

Combine all scraped data, enrich with details, and export to `raw.csv` with all required fields.

In [19]:
# Combine all DataFrames
all_dfs = []

if not wassit_df.empty:
    all_dfs.append(wassit_df)
    print(f"Added {len(wassit_df)} listings from Wassit")

if not lagence_df.empty:
    all_dfs.append(lagence_df)
    print(f"Added {len(lagence_df)} listings from L'agence MR")

if not opensooq_df.empty:
    all_dfs.append(opensooq_df)
    print(f"Added {len(opensooq_df)} listings from OpenSooq MR")

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    
    # Remove duplicates based on URL
    if "url" in df_all.columns:
        df_all = df_all.drop_duplicates(subset=["url"], keep="first")
    
    print(f"\nTotal unique listings: {len(df_all)}")
    print(f"Breakdown by source:")
    print(df_all["source"].value_counts())
    
    # Enrich with detail pages
    print("\n" + "="*50)
    print("ENRICHING WITH DETAIL PAGES")
    print("="*50)
    df_all = enrich_listings_with_details(df_all, session, delay_s=1.0, max_rows=ENRICH_MAX_ROWS)
    
    # Standardize column names and ensure all required columns exist
    required_columns = [
        "titre", "type_bien", "type_annonce", "prix", "surface_m2",
        "nb_chambres", "nb_salons", "nb_sdb", "quartier", "ville",
        "description", "source", "date_publication", "caracteristiques", "url"
    ]
    
    # Ensure all columns exist
    for col in required_columns:
        if col not in df_all.columns:
            df_all[col] = None
    
    # Reorder columns: required columns first, then any extras
    extra_cols = [c for c in df_all.columns if c not in required_columns]
    df_all = df_all[required_columns + extra_cols]
    
    # Clean numeric columns
    numeric_cols = ["prix", "surface_m2", "nb_chambres", "nb_salons", "nb_sdb"]
    for col in numeric_cols:
        if col in df_all.columns:
            df_all[col] = pd.to_numeric(df_all[col], errors="coerce")
    
    # Normalize property types
    if "type_bien" in df_all.columns:
        type_mapping = {
            "Maison": "Villa",
            "Riad": "Villa",
            "maison": "Villa",
            "riad": "Villa",
        }
        df_all["type_bien"] = df_all["type_bien"].replace(type_mapping)
    
    print(f"\n" + "="*50)
    print("FINAL SUMMARY")
    print("="*50)
    print(f"Total listings: {len(df_all)}")
    print(f"\nBreakdown by source:")
    print(df_all["source"].value_counts())
    
    if "type_bien" in df_all.columns:
        print(f"\nProperty types:")
        print(df_all["type_bien"].value_counts(dropna=False).head(10))
    
    if "type_annonce" in df_all.columns:
        print(f"\nListing types:")
        print(df_all["type_annonce"].value_counts(dropna=False))
    
    if "prix" in df_all.columns:
        print(f"\nPrice statistics:")
        print(df_all["prix"].describe())
    
    if "ville" in df_all.columns:
        print(f"\nCities:")
        print(df_all["ville"].value_counts(dropna=False).head(10))
    
    if "quartier" in df_all.columns:
        print(f"\nNeighborhoods:")
        print(df_all["quartier"].value_counts(dropna=False).head(10))
    
    # Display sample
    print(f"\n" + "="*50)
    print("SAMPLE DATA")
    print("="*50)
    display(df_all[required_columns].head(10))
    
    # Export to raw.csv with only required columns
    output_file = "raw.csv"
    
    # Select only the required columns in the correct order (including source and url)
    final_columns = [
        "titre", "type_bien", "type_annonce", "prix", "surface_m2",
        "nb_chambres", "nb_salons", "nb_sdb", "quartier", "ville",
        "description", "source", "date_publication", "caracteristiques", "url"
    ]
    
    # Create final DataFrame with only required columns
    df_final = df_all[final_columns].copy()
    
    # Clean up data before export
    # Replace None with empty string for CSV
    df_final = df_final.fillna("")
    
    # Ensure caracteristiques is a string (comma-separated)
    if "caracteristiques" in df_final.columns:
        df_final["caracteristiques"] = df_final["caracteristiques"].astype(str).replace("nan", "")
    
    df_final.to_csv(output_file, index=False, encoding="utf-8")
    print(f"\n✅ Exported to {output_file}")
    print(f"   Columns: {', '.join(final_columns)}")
    print(f"   Total rows: {len(df_final)}")
    
    # Show data quality summary
    print(f"\n=== Data Quality Summary ===")
    for col in final_columns:
        non_empty = (df_final[col] != "").sum()
        pct = (non_empty / len(df_final) * 100) if len(df_final) > 0 else 0
        print(f"  {col}: {non_empty}/{len(df_final)} ({pct:.1f}%)")
    
else:
    print("⚠️ No data collected from any site. Please check the selectors and site structure.")
    df_all = pd.DataFrame()

Added 20 listings from Wassit
Added 9 listings from L'agence MR

Total unique listings: 29
Breakdown by source:
source
wassit        20
lagence-mr     9
Name: count, dtype: int64

ENRICHING WITH DETAIL PAGES
Enriching 29 listings with detail pages...
✅ Enriched 29 listings with details.

FINAL SUMMARY
Total listings: 29

Breakdown by source:
source
wassit        20
lagence-mr     9
Name: count, dtype: int64

Property types:
type_bien
Appartement    20
Villa           9
Name: count, dtype: int64

Listing types:
type_annonce
Location    29
Name: count, dtype: int64

Price statistics:
count    2.000000e+01
mean     4.497965e+06
std      1.352133e+07
min      8.000000e+02
25%      4.500000e+04
50%      7.000000e+04
75%      1.418750e+06
max      6.000000e+07
Name: prix, dtype: float64

Cities:
ville
Nouakchott    29
Name: count, dtype: int64

Neighborhoods:
quartier
None          19
Arafat         9
Las Palmas     1
Name: count, dtype: int64

SAMPLE DATA


,titre,type_bien,type_annonce,prix,surface_m2,nb_chambres,nb_salons,nb_sdb,quartier,ville,description,source,date_publication,caracteristiques,url
0,Maison aux Lvelouje,Appartement,Location,14500000.0,NaN,NaN,NaN,NaN,None,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,33-42-27,climatisation,http://wassit.info/annonces/12434.html
1,"Machine de brique, parpaing, pavÃ©s autobloquant",Appartement,Location,NaN,400.0,NaN,NaN,NaN,None,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,None,climatisation,http://wassit.info/annonces/12406.html
2,Machine de fabrication de brique,Appartement,Location,NaN,NaN,NaN,NaN,NaN,None,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,None,climatisation,http://wassit.info/annonces/12405.html
3,Machine a parpaing,Appartement,Location,NaN,250.0,NaN,NaN,NaN,None,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,None,climatisation,http://wassit.info/annonces/12403.html
4,COMPLEXE A VENDRE,Appartement,Location,725000.0,1800.0,4.0,NaN,NaN,None,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,None,"garage, climatisation",http://wassit.info/annonces/12375.html
5,Un appartement meublÃ© Ã louer,Appartement,Location,150000.0,NaN,NaN,NaN,NaN,None,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,20-83-12,climatisation,http://wassit.info/annonces/12312.html
6,Ø¯Ø§Ø± Ø§Ù„Ù†Ø¹ÙŠÙ… Ø­Ù‰ 20 Ù‚Ø±Ø¨ Ø§Ù„Ù…Ø·Ø§Ø...,Appartement,Location,NaN,NaN,NaN,NaN,NaN,None,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,None,climatisation,http://wassit.info/annonces/12274.html
7,Ø¯Ø§Ø± Ø§Ù„Ù†Ø¹ÙŠÙ… Ø­Ù‰ 20 Ù‚Ø±Ø¨ Ø§Ù„Ù…Ø·Ø§Ø...,Appartement,Location,NaN,NaN,NaN,NaN,NaN,None,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,None,climatisation,http://wassit.info/annonces/12273.html
8,Terrain Ã vendre dubai-toujouninie-nouakchott,Appartement,Location,800.0,NaN,NaN,NaN,NaN,None,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,30-40-32,climatisation,http://wassit.info/annonces/12271.html
9,Dar anaiym,Appartement,Location,3500000.0,NaN,NaN,NaN,NaN,None,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,None,climatisation,http://wassit.info/annonces/12270.html



✅ Exported to raws.csv
   Columns: titre, type_bien, type_annonce, prix, surface_m2, nb_chambres, nb_salons, nb_sdb, quartier, ville, description, source, date_publication, caracteristiques, url
   Total rows: 29

=== Data Quality Summary ===
  titre: 29/29 (100.0%)
  type_bien: 29/29 (100.0%)
  type_annonce: 29/29 (100.0%)
  prix: 20/29 (69.0%)
  surface_m2: 3/29 (10.3%)
  nb_chambres: 10/29 (34.5%)
  nb_salons: 0/29 (0.0%)
  nb_sdb: 0/29 (0.0%)
  quartier: 10/29 (34.5%)
  ville: 29/29 (100.0%)
  description: 29/29 (100.0%)
  source: 29/29 (100.0%)
  date_publication: 11/29 (37.9%)
  caracteristiques: 29/29 (100.0%)
  url: 29/29 (100.0%)


In [20]:
# Combine all DataFrames
all_dfs = []

if not wassit_df.empty:
    all_dfs.append(wassit_df)
    print(f"Added {len(wassit_df)} listings from Wassit")

if not lagence_df.empty:
    all_dfs.append(lagence_df)
    print(f"Added {len(lagence_df)} listings from L'agence MR")

if not opensooq_df.empty:
    all_dfs.append(opensooq_df)
    print(f"Added {len(opensooq_df)} listings from OpenSooq MR")

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    
    # Clean and standardize
    if "price_mru" in df_all.columns:
        df_all["price_mru"] = pd.to_numeric(df_all["price_mru"], errors="coerce")
    
    # Remove duplicates based on URL
    if "url" in df_all.columns:
        df_all = df_all.drop_duplicates(subset=["url"], keep="first")
    
    # Ensure all expected columns exist
    expected_cols = ["source", "url", "title", "raw_text", "price_mru", "location"]
    for col in expected_cols:
        if col not in df_all.columns:
            df_all[col] = None
    
    # Reorder columns
    df_all = df_all[expected_cols + [c for c in df_all.columns if c not in expected_cols]]
    
    print(f"\nTotal unique listings: {len(df_all)}")
    print(f"\nBreakdown by source:")
    print(df_all["source"].value_counts())
    
    # Display sample
    display(df_all.head(20))
    
    # Export to raw.csv
    output_file = "raw.csv"
    df_all.to_csv(output_file, index=False, encoding="utf-8")
    print(f"\n✅ Exported to {output_file}")
    
    # Show summary statistics
    print("\n=== Summary Statistics ===")
    if "price_mru" in df_all.columns:
        print("\nPrice Statistics:")
        print(df_all["price_mru"].describe())
    
    if "location" in df_all.columns:
        print("\nTop Locations:")
        print(df_all["location"].value_counts().head(10))
else:
    print("⚠️ No data collected from any site. Please check the selectors and site structure.")
    df_all = pd.DataFrame()

Added 20 listings from Wassit
Added 9 listings from L'agence MR

Total unique listings: 29

Breakdown by source:
source
wassit        20
lagence-mr     9
Name: count, dtype: int64


,source,url,title,raw_text,price_mru,location
0,wassit,http://wassit.info/annonces/12434.html,Maison aux Lvelouje,14 500 000 UM Maison aux Lvelouje Ù†ÙˆØ§ÙƒØ´...,14500000.0,None
1,wassit,http://wassit.info/annonces/12406.html,"Machine de brique, parpaing, pavÃ©s autobloquant","Machine de brique, parpaing, pavÃ©s autobloqua...",NaN,Nouakchott
2,wassit,http://wassit.info/annonces/12405.html,Machine de fabrication de brique,Machine de fabrication de brique Nouakchott - ...,NaN,Nouakchott
3,wassit,http://wassit.info/annonces/12403.html,Machine a parpaing,Machine a parpaing Nouakchott - Mauritanie Vu ...,NaN,Nouakchott
4,wassit,http://wassit.info/annonces/12375.html,COMPLEXE A VENDRE,725 000 UM COMPLEXE A VENDRE Nouadhibou - Maur...,725000.0,Nouadhibou
5,wassit,http://wassit.info/annonces/12312.html,Un appartement meublÃ© Ã louer,150 000 UM Un appartement meublÃ© Ã louer Nou...,150000.0,Nouakchott
6,wassit,http://wassit.info/annonces/12274.html,Ø¯Ø§Ø± Ø§Ù„Ù†Ø¹ÙŠÙ… Ø­Ù‰ 20 Ù‚Ø±Ø¨ Ø§Ù„Ù…Ø·Ø§Ø...,Ø¯Ø§Ø± Ø§Ù„Ù†Ø¹ÙŠÙ… Ø­Ù‰ 20 Ù‚Ø±Ø¨ Ø§Ù„Ù…Ø·Ø§Ø...,NaN,None
7,wassit,http://wassit.info/annonces/12273.html,Ø¯Ø§Ø± Ø§Ù„Ù†Ø¹ÙŠÙ… Ø­Ù‰ 20 Ù‚Ø±Ø¨ Ø§Ù„Ù…Ø·Ø§Ø...,Ø¯Ø§Ø± Ø§Ù„Ù†Ø¹ÙŠÙ… Ø­Ù‰ 20 Ù‚Ø±Ø¨ Ø§Ù„Ù…Ø·Ø§Ø...,NaN,None
8,wassit,http://wassit.info/annonces/12271.html,Terrain Ã vendre dubai-toujouninie-nouakchott,800 UM Terrain Ã vendre dubai-toujouninie-nou...,800.0,Nouakchott
9,wassit,http://wassit.info/annonces/12270.html,Dar anaiym,3 500 000 UM Dar anaiym Ù†ÙˆØ§ÙƒØ´ÙˆØ· Vu 4420...,3500000.0,None



✅ Exported to raw.csv

=== Summary Statistics ===

Price Statistics:
count    1.900000e+01
mean     4.734384e+06
std      1.384931e+07
min      8.000000e+02
25%      5.000000e+04
50%      7.000000e+04
75%      2.112500e+06
max      6.000000e+07
Name: price_mru, dtype: float64

Top Locations:
location
Nouakchott    9
Nouadhibou    1
Name: count, dtype: int64


## Optional: Use Selenium for JavaScript-heavy sites

If the sites require JavaScript to load content, use Selenium as a fallback.

In [21]:
# Optional: Selenium fallback for JavaScript-heavy sites
USE_SELENIUM = True  # Set to True if sites need JavaScript

if USE_SELENIUM:
    try:
        from selenium import webdriver
        from selenium.webdriver.chrome.options import Options
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support.ui import WebDriverWait
        from selenium.webdriver.support import expected_conditions as EC
        from selenium.common.exceptions import TimeoutException
        
        def scrape_with_selenium(url: str, wait_time: int = 10) -> str:
            """Scrape a page using Selenium (for JavaScript-heavy sites)."""
            chrome_options = Options()
            chrome_options.add_argument("--headless=new")
            chrome_options.add_argument("--no-sandbox")
            chrome_options.add_argument("--disable-dev-shm-usage")
            chrome_options.add_argument("--window-size=1365,768")
            
            driver = webdriver.Chrome(options=chrome_options)
            driver.get(url)
            
            # Wait for content to load
            time.sleep(wait_time)
            
            html = driver.page_source
            driver.quit()
            return html
        
        print("Selenium is available. Set USE_SELENIUM = True to use it.")
    except ImportError:
        print("Selenium not installed. Install with: pip install selenium")
        print("Also install ChromeDriver: https://chromedriver.chromium.org/")
else:
    print("Selenium mode disabled. Using requests only.")

Selenium is available. Set USE_SELENIUM = True to use it.


## Regenerate raw.csv with All Required Columns

Run this cell to regenerate `raw.csv` with all the required fields:
- titre, type_bien, type_annonce, prix, surface_m2
- nb_chambres, nb_salons, nb_sdb, quartier, ville
- description, source, date_publication, caracteristiques, url

In [22]:
# Regenerate raw.csv with all required columns
# This cell will enrich listings and export to raw.csv with the correct format

# Check if we have listings data
if 'df_all' not in globals() or df_all.empty:
    print("⚠️ No listings data found. Please run cells 1-8 first to scrape listings.")
else:
    print(f"Found {len(df_all)} listings. Enriching with detail pages...")
    
    # Set enrichment limit (None = all, or use a number for testing)
    ENRICH_MAX_ROWS = None  # Change to 10 for quick testing
    
    # Enrich with detail pages
    if 'session' not in globals():
        session = make_session(verify_ssl=False)
    
    df_enriched = enrich_listings_with_details(df_all.copy(), session, delay_s=1.0, max_rows=ENRICH_MAX_ROWS)
    
    # Define required columns in correct order (including source and url)
    required_columns = [
        "titre", "type_bien", "type_annonce", "prix", "surface_m2",
        "nb_chambres", "nb_salons", "nb_sdb", "quartier", "ville",
        "description", "source", "date_publication", "caracteristiques", "url"
    ]
    
    # Ensure all columns exist
    for col in required_columns:
        if col not in df_enriched.columns:
            df_enriched[col] = None
    
    # Create final DataFrame with only required columns
    df_final = df_enriched[required_columns].copy()
    
    # Clean numeric columns
    numeric_cols = ["prix", "surface_m2", "nb_chambres", "nb_salons", "nb_sdb"]
    for col in numeric_cols:
        if col in df_final.columns:
            df_final[col] = pd.to_numeric(df_final[col], errors="coerce")
    
    # Normalize property types
    if "type_bien" in df_final.columns:
        type_mapping = {
            "Maison": "Villa",
            "Riad": "Villa",
            "maison": "Villa",
            "riad": "Villa",
        }
        df_final["type_bien"] = df_final["type_bien"].replace(type_mapping)
    
    # Replace NaN/None with empty strings for CSV
    df_final = df_final.fillna("")
    
    # Ensure caracteristiques is a string
    if "caracteristiques" in df_final.columns:
        df_final["caracteristiques"] = df_final["caracteristiques"].astype(str).replace("nan", "").replace("None", "")
    
    # Export to raw.csv
    output_file = "raw.csv"
    df_final.to_csv(output_file, index=False, encoding="utf-8")
    
    print(f"\n✅ Successfully exported to {output_file}")
    print(f"   Total listings: {len(df_final)}")
    print(f"   Columns: {', '.join(required_columns)}")
    
    # Show data quality summary
    print(f"\n=== Data Quality Summary ===")
    for col in required_columns:
        non_empty = (df_final[col] != "").sum()
        pct = (non_empty / len(df_final) * 100) if len(df_final) > 0 else 0
        print(f"  {col:20s}: {non_empty:3d}/{len(df_final):3d} ({pct:5.1f}%)")
    
    # Display sample
    print(f"\n=== Sample Data (first 5 rows) ===")
    display(df_final.head())

Found 29 listings. Enriching with detail pages...
Enriching 29 listings with detail pages...
✅ Enriched 29 listings with details.

✅ Successfully exported to raw.csv
   Total listings: 29
   Columns: titre, type_bien, type_annonce, prix, surface_m2, nb_chambres, nb_salons, nb_sdb, quartier, ville, description, source, date_publication, caracteristiques, url

=== Data Quality Summary ===
  titre               :  29/ 29 (100.0%)
  type_bien           :  29/ 29 (100.0%)
  type_annonce        :  29/ 29 (100.0%)
  prix                :  20/ 29 ( 69.0%)
  surface_m2          :   3/ 29 ( 10.3%)
  nb_chambres         :  10/ 29 ( 34.5%)
  nb_salons           :   0/ 29 (  0.0%)
  nb_sdb              :   0/ 29 (  0.0%)
  quartier            :  10/ 29 ( 34.5%)
  ville               :  29/ 29 (100.0%)
  description         :  29/ 29 (100.0%)
  source              :  29/ 29 (100.0%)
  date_publication    :  11/ 29 ( 37.9%)
  caracteristiques    :  29/ 29 (100.0%)
  url                 :  29/ 29 (100

,titre,type_bien,type_annonce,prix,surface_m2,nb_chambres,nb_salons,nb_sdb,quartier,ville,description,source,date_publication,caracteristiques,url
0,Maison aux Lvelouje,Appartement,Location,14500000.0,,,,,,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,33-42-27,climatisation,http://wassit.info/annonces/12434.html
1,"Machine de brique, parpaing, pavÃ©s autobloquant",Appartement,Location,,400.0,,,,,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,,climatisation,http://wassit.info/annonces/12406.html
2,Machine de fabrication de brique,Appartement,Location,,,,,,,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,,climatisation,http://wassit.info/annonces/12405.html
3,Machine a parpaing,Appartement,Location,,250.0,,,,,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,,climatisation,http://wassit.info/annonces/12403.html
4,COMPLEXE A VENDRE,Appartement,Location,725000.0,1800.0,4.0,,,,Nouakchott,Petites annonces gratuites en Mauritanie - - ...,wassit,,"garage, climatisation",http://wassit.info/annonces/12375.html
